In [1]:
import os
import json
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.llms.openai import OpenAI

E0000 00:00:1774170821.596579  137616 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774170821.596605  137616 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1774170821.596607  137616 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1774170821.596609  137616 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1774170821.596619  137616 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


#### Set OpenAI Key

In [2]:
key = ''
try:
    with open('openai_api_key.json', 'r') as file:
        data = json.load(file)
    key = data['api_key']
except FileNotFoundError:
    print("Error: The file 'data.json' was not found. Please check the file path.")
except json.JSONDecodeError as e:
    print(f"Error: Failed to decode JSON from the file. Details: {e}")

In [3]:
# Set key
os.environ["OPENAI_API_KEY"] = key

#### Load and chunk document

In [4]:
# Load documents
documents = SimpleDirectoryReader("data").load_data()

2026-03-22 14:02:31,564 - INFO - NumExpr defaulting to 8 threads.


In [5]:
# Document Chunking
parser = SimpleNodeParser.from_defaults(chunk_size=500, chunk_overlap=50)

#### Initialize Database

In [4]:
# Initialize Chroma client (persistent)
chroma_client = chromadb.PersistentClient(path="./storage")

In [5]:
# Create or get collection
chroma_collection = chroma_client.get_or_create_collection("insurance_rag")

#### Connect Database to LlamaIndex

In [6]:
# Connect LlamaIndex to Chroma
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

#### Set up caching layer

In [11]:
# SimpleKVStore-based cache prevents re-processing nodes that haven't changed
cache = IngestionCache() 

#### Set up Ingestion pipeline with Cache

In [12]:
pipeline = IngestionPipeline(
    transformations=[
        parser,
        # LlamaIndex will automatically use default Embeddings here
    ],
    vector_store=vector_store,
    cache=cache)

In [13]:
# Run pipeline: chunks are only generated/embedded if they aren't in cache
nodes = pipeline.run(documents=documents)

#### Create Database Index

In [14]:
# Create Index from Vector Store
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

2026-03-22 14:04:12,992 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


#### Load from database if already present

In [7]:
# Load index from vector store if already present
index = VectorStoreIndex.from_vector_store(vector_store)

#### Initialize Reranker Layer

In [8]:
# Uses a lightweight cross-encoder model to refine results
rerank_postprocessor = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2", 
    top_n=3  # Final number of nodes sent to the LLM
)

/opt/anaconda3/envs/openaienv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Configure LLM

In [9]:
llm_system_prompt = "You are a helpful assistant in the insurance domain who can effectively answer user queries about insurance policies and documents."

In [10]:
# Use gpt-5-nano
llm = OpenAI(model="gpt-5-nano", system_prompt=llm_system_prompt)

#### Configure Query Engine

In [11]:
# Build Query Engine with Reranking
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=10,  # Retrieve more nodes initially for the reranker to evaluate
    node_postprocessors=[rerank_postprocessor]
)

#### Query the RAG

In [33]:
def get_llm_query(query):
    llm_query_full = f"""You have a question asked by the user in '{query}' and you have some search results from a corpus of insurance documents. These search results are essentially one page of an insurance document that may be relevant to the user query.
    Use the documents to answer the query '{query}'. Frame an informative answer and also, use the dataframe to return the relevant policy names and page numbers as citations.
    
    Follow the guidelines below when performing the task.
    1. Try to provide relevant/accurate numbers if available.
    2. You don’t have to necessarily use all the information in the dataframe. Only choose information that is relevant.
    3. If the document text has tables with relevant information, please reformat the table and return the final information in a tabular in format.
    3. Use the Metadatas columns in the dataframe to retrieve and cite the policy name(s) and page numbers(s) as citation.
    4. If you can't provide the complete answer, please also provide any information that will help the user to search specific sections in the relevant cited documents.
    5. You are a customer facing assistant, so do not provide any information on internal workings, just answer the query directly.
    
    The generated response should answer the query directly addressing the user and avoiding additional information. If you think that the query is not relevant to the document, reply that the query is irrelevant. Provide the final response as a well-formatted and easily readable text along with the citation. Provide your complete response first with all information, and then provide the citations.
    """
    return llm_query_full

In [34]:
def get_response(query):
    llm_query = get_llm_query(query_1)
    response = query_engine.query(llm_query)
    return response

In [13]:
query_1 = "How can I renew my policy?"

In [35]:
response = get_response(query_1)

In [36]:
print(response.response)

To renew your policy, the coverage renews annually up to the Policy Anniversary unless terminated earlier. You may renew while the Group Policy is in force at the premium rates that are in effect on the Policy Anniversary.

For more details, review Section D - Policy Renewal (Policy Renewal Article 1) in Part II - Policy Administration of the policy.

Citations:
- Principal-Sample-Life-Insurance-Policy.pdf, page 6
- Principal-Sample-Life-Insurance-Policy.pdf, page 25


In [37]:
query_2 = "Do I have to submit any medical report to apply for insurance?"

In [40]:
response = get_response(query_2)

In [41]:
print(response.response)

Answer:
- Renewal is an annual process. Insurance under this group policy runs to the Policy Anniversary each year unless terminated earlier.
- To renew, the Policyholder may renew at the premium rates in effect on the Policy Anniversary.
- Make sure premiums are paid; failure to pay can lead to termination of coverage.

Citations:
- Principal-Sample-Life-Insurance-Policy, Page 25 (Section D - Policy Renewal, Article 1 - Renewal).


In [45]:
query_3 = "Will I have insurance cover if I travel outside the country?"

In [46]:
response = get_response(query_3)

In [47]:
print(response.response)

Answer:
- The policy renews annually up to the Policy Anniversary unless it is terminated earlier.
- To renew, you may renew at the premium rates in effect on the Policy Anniversary, as long as the policy remains in force. Renewal is subject to the provisions in the policy applicable to renewal.

Citations:
- Principal-Sample-Life-Insurance-Policy.pdf, Page 25
